# Klasifikasi DemogPairs Menggunakan ViT (Emosi dan Wajah) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(emotion_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

  0%|          | 0/10800 [00:00<?, ?it/s]

  7%|▋         | 781/10800 [00:00<00:01, 7801.53it/s]

 14%|█▍        | 1562/10800 [00:00<00:01, 7786.71it/s]

 22%|██▏       | 2399/10800 [00:00<00:01, 8048.64it/s]

 30%|██▉       | 3204/10800 [00:00<00:00, 7933.06it/s]

...

 82%|████████▏ | 8840/10800 [00:01<00:00, 7225.35it/s]

 89%|████████▊ | 9569/10800 [00:01<00:00, 7198.50it/s]

 96%|█████████▋| 10401/10800 [00:01<00:00, 7521.44it/s]

100%|██████████| 10800/10800 [00:01<00:00, 7690.64it/s]

Jumlah fitur per gambar: 1536


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_rf_vit-emotion-face_",
    results_path="results/demogpairs_rf_vit-emotion-face_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.8685185185185185
Precision : 0.868917135945467
Recall    : 0.8685185185185184
F1 Score  : 0.868240801565065
               precision    recall  f1-score   support

Asian_Females     0.8232    0.8278    0.8255       360
  Asian_Males     0.8270    0.8500    0.8384       360
Black_Females     0.8815    0.8056    0.8418       360
  Black_Males     0.9155    0.9028    0.9091       360
White_Females     0.8861    0.8861    0.8861       360
  White_Males     0.8802    0.9389    0.9086       360

     accuracy                         0.8685      2160
    macro avg     0.8689    0.8685    0.8682      2160
 weighted avg     0.8689    0.8685    0.8682      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9416666666666667,0.8232044198895028,0.8277777777777777,0.8254847645429362,360
Asian_Males,0.9453703703703704,0.827027027027027,0.85,0.8383561643835615,360
Black_Females,0.9495370370370371,0.8814589665653495,0.8055555555555556,0.841799709724238,360
Black_Males,0.9699074074074074,0.9154929577464789,0.9027777777777778,0.9090909090909091,360
White_Females,0.962037037037037,0.8861111111111111,0.8861111111111111,0.8861111111111111,360
White_Males,0.9685185185185186,0.8802083333333334,0.9388888888888889,0.9086021505376345,360


Confusion matrix saved: images\cm_rf_vit-emotion-face_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               298                24                21                 0                13                 4
         Asian_Males                22               306                 2                 8                 0                22
       Black_Females                17                10               290                18                22                 3
         Black_Males                 0                14                 6               325                 0                15
       White_Females                23                 5                10                 1               319                 2
         White_Males                 2                11                 0                 3                 6               338


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-emotion-face_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8685185185185185,0.868240801565065,0.868917135945467,0.8685185185185184,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-emotion-face_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 6466.0,
 'days': 0,
 'hours': 1,
 'minutes': 47,
 'seconds': 46.0,
 'text': '0 hari 1 jam 47 menit 46.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 25052.0,
 'days': 0,
 'hours': 6,
 'minutes': 57,
 'seconds': 32.0,
 'text': '0 hari 6 jam 57 menit 32.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8866,0.875,0.8686,0.8814,0.8744,0.8772,0.8768,0.8776,0.8772,19.5185
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8866,0.8744,0.8704,0.8808,0.8733,0.8771,0.8767,0.8775,0.8771,19.9829
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8848,0.8779,0.8709,0.8727,0.8779,0.8769,0.8765,0.8773,0.8769,19.178
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.886,0.8744,0.8605,0.8808,0.8808,0.8765,0.8761,0.8766,0.8765,20.6288
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.8733,0.8634,0.8507,0.8623,0.8611,0.8622,0.8612,0.8629,0.8622,8.4222
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.8744,0.8623,0.8513,0.8594,0.8628,0.862,0.8611,0.8625,0.862,8.4268
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.8738,0.8565,0.849,0.8623,0.8634,0.861,0.8599,0.8615,0.861,8.5768
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.8738,0.8542,0.849,0.8617,0.8646,0.8606,0.8596,0.8611,0.8606,6.2923
